# **Project 2 modified implementation**

# Basic Project overview

This notebook implements LSTM-based POS tagging using a locally extracted Penn Treebank directory. It reads the WSJ POS files directly from `./treebank_3/tagged/pos/wsj/` (relative to the notebook's **working directory**). It does not require Google Colab or an uploaded `.tgz` archive.

## Notes for running locally

1. Install Python and PyTorch, plus Jupyter if you want to run the `.ipynb` notebook directly.
2. Extract the course-provided Treebank archive so that `treebank_3/tagged/pos/wsj/` exists alongside the notebook. Keep the Treebank files private to the course.
3. Start Jupyter from the project directory, or change `TREEBANK_ROOT` below if your working directory differs.
4. Run all code cells from top to bottom. The training/test split and model parameters remain as in the uploaded notebook.
5. A CUDA-enabled GPU is used automatically if your PyTorch installation supports one. Otherwise, the code runs on CPU.
6. Edit `USER_SENTENCE` to tag a custom sentence. The last experiment section trains a separate stacked bidirectional LSTM.


# Local Penn Treebank directory

The extracted archive must have `.pos` files under `./treebank_3/tagged/pos/wsj/`. The path is relative to Python's current working directory, usually the directory from which you started Jupyter.


In [ ]:
from pathlib import Path

TREEBANK_ROOT = Path('./treebank_3')
WSJ_POS_DIR = TREEBANK_ROOT / 'tagged' / 'pos' / 'wsj'

if not WSJ_POS_DIR.is_dir():
    raise FileNotFoundError(
        f'Penn Treebank WSJ POS directory not found: {WSJ_POS_DIR.resolve()}\n'
        f'Current working directory: {Path.cwd()}\n'
        'Expected: ./treebank_3/tagged/pos/wsj/00/*.pos etc. '
        'Extract the archive or update TREEBANK_ROOT.'
    )

pos_file_count = sum(1 for _ in WSJ_POS_DIR.rglob('*.pos'))
if pos_file_count == 0:
    raise RuntimeError(f'No .pos files found under {WSJ_POS_DIR.resolve()}')

print('Working directory:', Path.cwd())
print('Penn Treebank POS directory:', WSJ_POS_DIR.resolve())
print('POS files found:', pos_file_count)


# **1.  Config and Imports**

In [ ]:
import re, random, copy
from collections import Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
#Select hardware usage
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Distinct WSJ sections: Kept small to not use too much computation uncomment section below or change ranges for larger sets
'''
TRAIN_SECTIONS = range(0, 5)  # 00, 01, 02, 03, 04
VAL_SECTIONS   = range(5, 6)  # 05
TEST_SECTIONS  = range(6, 7)  # 06
MAX_FILES_PER_SECTION = 30  # None = use every .pos file in each section
'''
TRAIN_SECTIONS = range(0, 19) # 00-18
VAL_SECTIONS   = range(19, 22) #19 - 21
TEST_SECTIONS  = range(22, 25) #22 -24
MAX_FILES_PER_SECTION = None


#Hyperparameters
MIN_WORD_FREQ = 2 #exclude words that appear exactly once
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 1
BIDIRECTIONAL = False #Change to True for Bi-LSTM
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.002
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

# Parsing

In [ ]:
def parse_pos(text): # produce input sequences
    sentences, pairs = [], []
    for item in text.split():
        if item in {'[', ']'} or item.startswith('==='):
            continue
        if '/' not in item:
            continue
        word, tag = item.rsplit('/', 1)
        if not word or not tag:
            continue
        tag = tag.split('|', 1)[0]  # ambiguous composite labels: use first POS tag
        pairs.append((word, tag))
        if word in {'.', '?', '!'} and tag == '.':
            sentences.append(([w for w, _ in pairs], [t for _, t in pairs]))
            pairs = []
    if pairs:
        sentences.append(([w for w, _ in pairs], [t for _, t in pairs]))
    return sentences


def load_wsj_splits(pos_dir):
    sections = set(TRAIN_SECTIONS) | set(VAL_SECTIONS) | set(TEST_SECTIONS)
    assert not (set(TRAIN_SECTIONS) & set(VAL_SECTIONS) or
                set(TRAIN_SECTIONS) & set(TEST_SECTIONS) or
                set(VAL_SECTIONS) & set(TEST_SECTIONS)), 'Split sections must not overlap'

    files_by_section = {section: [] for section in sections}
    for sec in sorted(sections):
        section_dir = pos_dir / f'{sec:02d}'
        if not section_dir.is_dir():
            raise FileNotFoundError(f'Missing WSJ section directory: {section_dir.resolve()}')
        paths = sorted(section_dir.glob('*.pos'))
        if MAX_FILES_PER_SECTION is not None:
            paths = paths[:MAX_FILES_PER_SECTION]
        for path in paths:
            contents = path.read_text(encoding='latin-1')
            files_by_section[sec].append((str(path), parse_pos(contents)))

    def collect(section_ids):
        return [sentence for sec in section_ids for _, sentences in files_by_section[sec]
                for sentence in sentences]

    splits = (collect(TRAIN_SECTIONS), collect(VAL_SECTIONS), collect(TEST_SECTIONS))
    for label, section_ids, data in zip(['Train', 'Validation', 'Test'],
                                        [TRAIN_SECTIONS, VAL_SECTIONS, TEST_SECTIONS], splits):
        used_files = [name for sec in section_ids for name, _ in files_by_section[sec]]
        print(f'{label}: sections {list(section_ids)}, {len(used_files)} files, '
              f'{len(data)} sentences, {sum(len(words) for words, _ in data)} tokens')
        print('  Sample files:', used_files[:2])

    if any(not split for split in splits):
        raise RuntimeError('A dataset split is empty; check directory layout and section selection.')
    return splits, files_by_section


(train_data, val_data, test_data), files_by_section = load_wsj_splits(WSJ_POS_DIR)
print('Example words:', train_data[0][0][:20])
print('Example tags: ', train_data[0][1][:20])
assert all(len(words) == len(tags) and len(words) > 0
           for split in (train_data, val_data, test_data) for words, tags in split)


# Vocab and handling unknown words

Build word/tag dict from training data, <PAD> pads variable length sequences and <UNK> represents words that are absent/rare in training vocab. lower case words when indexing

In [ ]:
word_counts = Counter(word.lower() for words, _ in train_data for word in words)
word_to_ix = {'<PAD>': 0, '<UNK>': 1}
for word, count in sorted(word_counts.items()):
    if count >= MIN_WORD_FREQ:
        word_to_ix[word] = len(word_to_ix)

tag_to_ix = {tag: index for index, tag in enumerate(sorted({tag for _, tags in train_data for tag in tags}))}
ix_to_tag = {index: tag for tag, index in tag_to_ix.items()}
missing = sorted({t for data in (val_data, test_data) for _, tags in data for t in tags} - set(tag_to_ix))
if missing:
    raise RuntimeError(f'Tags only in validation/test: {missing}. Increase training section coverage.')

for label, data in [('Validation', val_data), ('Test', test_data)]:
    tokens = [w.lower() for words, _ in data for w in words]
    oov = sum(w not in word_to_ix for w in tokens) / max(1, len(tokens)) #tokens mapped to unknown/ total tokens
    print(f'{label} OOV rate: {oov:.2%}')
print('Word vocabulary size:', len(word_to_ix), '| POS tags:', len(tag_to_ix))
print('Tags:', sorted(tag_to_ix))

# Batched LSTM Tagger


In [ ]:
class POSTagDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, index):
        words, tags = self.pairs[index]
        x = torch.tensor([word_to_ix.get(w.lower(), word_to_ix['<UNK>']) for w in words], dtype=torch.long)
        y = torch.tensor([tag_to_ix[t] for t in tags], dtype=torch.long)
        return x, y


def collate_pos(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    xpad = nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=word_to_ix['<PAD>'])
    ypad = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=-100)
    return xpad, ypad, lengths

train_loader = DataLoader(POSTagDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_pos)
val_loader = DataLoader(POSTagDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pos)
test_loader = DataLoader(POSTagDataset(test_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_pos)

class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, tag_count, embedding_dim, hidden_dim, num_layers=1, bidirectional=False):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=bidirectional)
        self.hidden2tag = nn.Linear(hidden_dim * (2 if bidirectional else 1), tag_count)

    def forward(self, x, lengths):
        embeds = self.word_embeddings(x)
        packed = nn.utils.rnn.pack_padded_sequence(embeds, lengths.cpu(),
                                                    batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed)
        padded, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True,
                                                     total_length=x.size(1))
        return self.hidden2tag(padded)


def new_model(bidirectional=False, num_layers=1): #also used later to experiment with bidrectiontal and multi layer LSTM
    return LSTMTagger(len(word_to_ix), len(tag_to_ix), EMBEDDING_DIM, HIDDEN_DIM,
                      num_layers=num_layers, bidirectional=bidirectional).to(DEVICE)

model = new_model(BIDIRECTIONAL, NUM_LAYERS)
print(model)

# Train and validation

Cross-entropy ignores padded targets (-100). Epoch with higheest validation accuracy selected. Held-out test set cannot be used to pick model settings

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = correct = total = 0
    with torch.set_grad_enabled(is_train):
        for x, y, lengths in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            scores = model(x, lengths)
            loss = criterion(scores.reshape(-1, scores.size(-1)), y.reshape(-1))
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            active = y != -100
            n = active.sum().item()
            total_loss += loss.item() * n
            correct += ((scores.argmax(-1) == y) & active).sum().item()
            total += n
    return total_loss / total, correct / total


def fit_model(model, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_accuracy = -1
    best_parameters = None
    history = []
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader)
        history.append((train_loss, train_acc, val_loss, val_acc))
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            best_parameters = copy.deepcopy(model.state_dict())
        print(f'Epoch {epoch}/{epochs}: train loss={train_loss:.4f}, acc={train_acc:.2%}; '
              f'val loss={val_loss:.4f}, acc={val_acc:.2%}')
    model.load_state_dict(best_parameters)
    return history, best_accuracy

history, best_val_acc = fit_model(model)
print(f'Best validation accuracy: {best_val_acc:.2%}')

# Evaluate on held-out test set

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader)
print(f'Held-out test accuracy: {test_acc:.2%} | test loss: {test_loss:.4f}')
print('Test tokens:', sum(len(words) for words, _ in test_data))

# Apply to sentence
Edit variable USER_SENTENCE to apply tagger to specific sentence

In [ ]:
def tokenize(sentence): #reg Expression tokenization
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+(?:[.,][0-9]+)*|[^\w\s]", sentence)

@torch.no_grad() #disable gradient recording to reduce computation and mem
def predict_tags(words, trained_model=model):
    if not words:
        return []
    trained_model.eval() #switch to eval mode
    ids = [word_to_ix.get(w.lower(), word_to_ix['<UNK>']) for w in words]
    x = torch.tensor([ids], device=DEVICE)
    scores = trained_model(x, torch.tensor([len(ids)]))
    return [ix_to_tag[i] for i in scores.argmax(-1)[0].tolist()]

#EDIT THIS SENTENCE ------------------------
USER_SENTENCE = 'This is a common sentence that is used right here.'
#USER_SENTENCE = 'The Rambunctious Rambodius student tagged this sentence.' #Uncomment if you want to see unknowns

words = tokenize(USER_SENTENCE)
tags = predict_tags(words)
print(f'{"Word":<22} {"Predicted POS":<15} Unknown?')
print('-' * 50)
for word, tag in zip(words, tags):
    print(f'{word:<22} {tag:<15} {word.lower() not in word_to_ix}')

# Block for experiments with architecture changes
Modify EXPERIMENT_BIDIRECTIONAL and NUM_LAYERS to see changes in accuracy for Bi-LSTM and/or stacked layers

In [ ]:
EXPERIMENT_BIDIRECTIONAL = True
EXPERIMENT_NUM_LAYERS = 3
experiment_model = new_model(EXPERIMENT_BIDIRECTIONAL, EXPERIMENT_NUM_LAYERS)
experiment_history, experiment_val_acc = fit_model(experiment_model)
print('Baseline best validation accuracy:', f'{best_val_acc:.2%}')
print('Experiment best validation accuracy:', f'{experiment_val_acc:.2%}')
print('Difference (percentage points):', round(100 * (experiment_val_acc - best_val_acc), 2))
# To use the experiment model for custom tagging, call: predict_tags(words, experiment_model)

# HyperParam choices

***Note that these choices were made with the larger training/validation/test sets in mind and performance may change with change in set size***

Bi-Directional

3 layers

EPOCHS = 5, doesn't seem to increase acc after 5 by a substantial amount an eventually starts dropping after 7
